# Hybrid Numerical–Reasoning Predictive Maintenance Framework  
## NASA Turbofan Engine (C-MAPSS) Dataset

**Author:** Jathin Sai Kodali  

This notebook demonstrates an end-to-end research workflow for predictive maintenance using the NASA C-MAPSS turbofan engine degradation dataset:
1. Data loading and preprocessing  
2. Remaining Useful Life (RUL) label generation  
3. Exploratory data analysis and visualization  
4. Sequence preparation for deep learning  
5. LSTM-based RUL prediction  
6. Evaluation and plots  
7. Reasoning layer for maintenance recommendations


## Notes before running
- Place the NASA C-MAPSS dataset files in the same folder as this notebook, especially:
  - `train_FD001.txt`
  - `test_FD001.txt`
  - `RUL_FD001.txt`
- This notebook is designed for **FD001** first because it is the simplest subset and best for demonstration.
- You can later extend the same pipeline to FD002, FD003, and FD004.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 1. Load dataset

In [ ]:
DATA_DIR = "."

train_path = os.path.join(DATA_DIR, "train_FD001.txt")
test_path = os.path.join(DATA_DIR, "test_FD001.txt")
rul_path = os.path.join(DATA_DIR, "RUL_FD001.txt")

print("Train exists:", os.path.exists(train_path))
print("Test exists:", os.path.exists(test_path))
print("RUL exists:", os.path.exists(rul_path))

In [ ]:
index_names = ['engine_id', 'cycle']
setting_names = ['op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_names = [f'sensor_{i}' for i in range(1, 22)]
columns = index_names + setting_names + sensor_names

def load_cmapps_file(path):
    df = pd.read_csv(path, sep=r"\s+", header=None)
    df = df.iloc[:, :26]
    df.columns = columns
    return df

train_df = load_cmapps_file(train_path)
test_df = load_cmapps_file(test_path)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None)
rul_df.columns = ["true_rul"]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", rul_df.shape)

train_df.head()

## 2. Generate Remaining Useful Life (RUL) labels

In [ ]:
max_cycle_train = train_df.groupby("engine_id")["cycle"].max().reset_index()
max_cycle_train.columns = ["engine_id", "max_cycle"]

train_df = train_df.merge(max_cycle_train, on="engine_id", how="left")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]

train_df[["engine_id", "cycle", "max_cycle", "RUL"]].head(10)

## 3. Basic dataset understanding

In [ ]:
print("Number of training engines:", train_df["engine_id"].nunique())
print("Number of test engines:", test_df["engine_id"].nunique())
print("\nDescriptive statistics:")
train_df.describe().T.head(10)

## 4. Exploratory visualizations

In [ ]:
# RUL degradation trend for one sample engine
sample_engine = 1
engine_data = train_df[train_df["engine_id"] == sample_engine]

plt.figure()
plt.plot(engine_data["cycle"], engine_data["RUL"])
plt.title(f"RUL Degradation Over Time - Engine {sample_engine}")
plt.xlabel("Cycle")
plt.ylabel("RUL")
plt.grid(True)
plt.show()

In [ ]:
# Plot a few important sensors for one engine
selected_sensors = ["sensor_2", "sensor_3", "sensor_4", "sensor_11"]

for sensor in selected_sensors:
    plt.figure()
    plt.plot(engine_data["cycle"], engine_data[sensor])
    plt.title(f"{sensor} Trend Over Cycles - Engine {sample_engine}")
    plt.xlabel("Cycle")
    plt.ylabel(sensor)
    plt.grid(True)
    plt.show()

In [ ]:
# Correlation heatmap using pandas + matplotlib only
corr_cols = setting_names + sensor_names + ["RUL"]
corr_matrix = train_df[corr_cols].corr()

plt.figure(figsize=(14, 10))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 5. Feature selection and normalization

In [ ]:
# Remove near-constant sensors often found in FD001
drop_sensors = ["sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19"]
feature_cols = [c for c in setting_names + sensor_names if c not in drop_sensors]

print("Number of selected features:", len(feature_cols))
print(feature_cols)

In [ ]:
scaler = MinMaxScaler()

train_df_scaled = train_df.copy()
test_df_scaled = test_df.copy()

train_df_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

train_df_scaled.head()

## 6. Prepare sequences for LSTM

In [ ]:
sequence_length = 30

def create_train_sequences(df, seq_len, feature_columns, target_col="RUL"):
    X, y = [], []
    for engine_id in df["engine_id"].unique():
        engine_df = df[df["engine_id"] == engine_id].sort_values("cycle")
        feature_array = engine_df[feature_columns].values
        target_array = engine_df[target_col].values

        for i in range(len(engine_df) - seq_len):
            X.append(feature_array[i:i + seq_len])
            y.append(target_array[i + seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_train_sequences(train_df_scaled, sequence_length, feature_cols)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
def create_test_sequences(test_df, rul_df, seq_len, feature_columns):
    X_test, y_test = [], []

    max_cycle_test = test_df.groupby("engine_id")["cycle"].max().reset_index()
    max_cycle_test.columns = ["engine_id", "max_cycle"]
    rul_df_local = rul_df.copy()
    rul_df_local["engine_id"] = np.arange(1, len(rul_df_local) + 1)

    truth_df = max_cycle_test.merge(rul_df_local, on="engine_id", how="left")
    truth_df["failure_cycle"] = truth_df["max_cycle"] + truth_df["true_rul"]

    test_with_truth = test_df.merge(truth_df[["engine_id", "failure_cycle"]], on="engine_id", how="left")
    test_with_truth["RUL"] = test_with_truth["failure_cycle"] - test_with_truth["cycle"]

    for engine_id in test_with_truth["engine_id"].unique():
        engine_df = test_with_truth[test_with_truth["engine_id"] == engine_id].sort_values("cycle")
        feature_array = engine_df[feature_columns].values
        target_array = engine_df["RUL"].values

        if len(engine_df) >= seq_len:
            X_test.append(feature_array[-seq_len:])
            y_test.append(target_array[-1])

    return np.array(X_test), np.array(y_test), test_with_truth

X_test, y_test, test_with_truth = create_test_sequences(test_df_scaled, rul_df, sequence_length, feature_cols)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## 7. Build and train the LSTM model

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

## 8. Training performance plot

In [ ]:
plt.figure()
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True)
plt.show()

## 9. Predict on test set

In [ ]:
y_pred = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

In [ ]:
results_df = pd.DataFrame({
    "Engine_ID": np.arange(1, len(y_test) + 1),
    "Actual_RUL": y_test,
    "Predicted_RUL": y_pred,
    "Absolute_Error": np.abs(y_test - y_pred)
})

results_df.head(10)

## 10. Prediction visualizations

In [ ]:
plt.figure()
plt.plot(results_df["Actual_RUL"].values, label="Actual RUL")
plt.plot(results_df["Predicted_RUL"].values, label="Predicted RUL")
plt.title("Actual vs Predicted RUL Across Test Engines")
plt.xlabel("Test Engine Index")
plt.ylabel("RUL")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
plt.scatter(results_df["Actual_RUL"], results_df["Predicted_RUL"])
plt.title("Actual vs Predicted RUL Scatter Plot")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.grid(True)
plt.show()

In [ ]:
top_error_df = results_df.sort_values("Absolute_Error", ascending=False).head(10)
top_error_df

## 11. Reasoning layer for maintenance intelligence

In [ ]:
def get_risk_level(rul):
    if rul > 100:
        return "LOW"
    elif rul > 50:
        return "MODERATE"
    else:
        return "CRITICAL"

def get_recommendation(rul):
    if rul > 100:
        return "Engine operating normally. Continue routine monitoring."
    elif rul > 50:
        return "Engine shows degradation. Schedule inspection and detailed diagnostics."
    else:
        return "Engine is near failure. Immediate maintenance or replacement is required."

def get_reasoning_report(predicted_rul):
    return {
        "Predicted_RUL": round(float(predicted_rul), 2),
        "Risk_Level": get_risk_level(predicted_rul),
        "Recommendation": get_recommendation(predicted_rul)
    }

In [ ]:
# Show reasoning output for first 10 engines
reasoning_outputs = [get_reasoning_report(v) for v in y_pred[:10]]
pd.DataFrame(reasoning_outputs)

In [ ]:
# Attach reasoning output to final results
results_df["Risk_Level"] = results_df["Predicted_RUL"].apply(get_risk_level)
results_df["Recommendation"] = results_df["Predicted_RUL"].apply(get_recommendation)

results_df.head(15)

## 12. Save trained model and results

In [ ]:
model.save("lstm_rul_fd001.h5")
results_df.to_csv("fd001_prediction_results.csv", index=False)

print("Saved:")
print("- lstm_rul_fd001.h5")
print("- fd001_prediction_results.csv")

## 13. Final conclusion

In [ ]:
print("""
Conclusion:
1. The NASA C-MAPSS FD001 dataset was successfully loaded and preprocessed.
2. Remaining Useful Life (RUL) labels were generated for supervised learning.
3. Sensor degradation patterns were visualized to support research analysis.
4. An LSTM model was trained to estimate engine RUL.
5. Model performance was evaluated using RMSE, MAE, and R2.
6. A reasoning layer converted numerical outputs into actionable maintenance intelligence.
7. This hybrid framework bridges the gap between prediction accuracy and real-world maintenance decision support.
""")